# 训练演示

此notebook用于快速启动训练和调试代码

In [1]:
# 导入模块

import os, re
import argparse
import torch
import datetime
import numpy as np
import random
import multiprocessing
import wandb
from agents import QCPO, QPO
from envs import RiskSensitiveEnv

### QPO


In [ ]:
# 定义参数
class SimpleArgs:
    def __init__(self):
        
        self.env_name = 'RiskSensitiveEnv'
        self.log_interval = 100
        self.est_interval = 100
        self.q_alpha = 0.25
        self.emb_dim = [8,8]
        self.max_episode = 100000  # 测试用，设置较小值
        self.init_std = np.sqrt(1e-1)
        self.gamma = 0.99
        self.algo_name = 'QPO'
        self.seed = 0

        # 学习率参数
        self.theta_a = (10000**0.9) * 1e-3
        self.theta_b = 10000
        self.theta_c = 0.9
        self.q_a = (10000**0.6) * 1e-2
        self.q_b = 10000
        self.q_c = 0.6

        
        # 设备和路径
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        # timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        # self.path = os.path.join('./logs/toy_env/test_qcpo', f'run_{timestamp}')
        # os.makedirs(self.path, exist_ok=True)



args = SimpleArgs()
random.seed(args.seed) 
np.random.seed(args.seed) 
torch.manual_seed(args.seed)


#env = AssetEnv(n=10)
env = RiskSensitiveEnv(n=10)
agent_QPO = QPO(args, env)

# 启动训练
agent_QPO.train()

# 结束wandb监控
wandb.finish()

### QCPO

In [ ]:
# 定义参数
class SimpleArgs:
    def __init__(self):
        
        self.env_name = 'RiskSensitiveEnv'
        self.log_interval = 100
        self.est_interval = 100
        self.q_alpha = 0.25
        self.emb_dim = [8,8]
        self.max_episode = 1000  # 测试用，设置较小值
        self.init_std = np.sqrt(1e-1)
        self.gamma = 0.99
        self.algo_name = 'QCPO'
        self.seed = 0

        # 学习率参数
        self.theta_a = (10000**0.9) * 1e-3
        self.theta_b = 10000
        self.theta_c = 0.9
        self.q_a = (10000**0.6) * 1e-2
        self.q_b = 10000
        self.q_c = 0.6
        self.lambda_a = 0.3                   # 如果不设置，默认使用q_est的参数
        self.lambda_b = 5000                 # 如果不设置，默认使用q_est的参数
        self.lambda_c = 0.6                   # 如果不设置，默认使用q_est的参数


        self.outer_interval = 10  # 控制每几个循环更新一次lmabda，因为更新lambda时需要估计期望和分位数 
        self.density_estimate = False # 是否使用密度估计版本（true使用密度估计，false使用nu）
        self.h_n = 0.01               # 密度估计版本的分位数间隔（仅在density_estimate=true时使用）
        self.nu = 1               # 仅当self.density_estimate=False时使用
        self.quantile_threshold = 4
              
        # 设备和路径
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        # timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        # self.path = os.path.join('./logs/toy_env/test_qcpo', f'run_{timestamp}')
        # os.makedirs(self.path, exist_ok=True)



args = SimpleArgs()
random.seed(args.seed) 
np.random.seed(args.seed) 
torch.manual_seed(args.seed)


#env = AssetEnv(n=10)
env = RiskSensitiveEnv(n=10)
agent_QCPO = QCPO(args, env)

# 启动训练
agent_QCPO.train()

# 结束wandb监控
wandb.finish()

### DQC-AC


In [ ]:
# ===================== DQC-AC 训练 =====================
# DQC-AC: Distributional Quantile-Constrained Actor-Critic
# 使用密度估计的约束优化方法，维护三个分位数点 Q(α-δ), Q(α), Q(α+δ)

from agents import DQCAC

# 定义参数
class SimpleArgs:
    def __init__(self):
        
        self.env_name = 'RiskSensitiveEnv'
        self.log_interval = 100
        self.est_interval = 100
        self.q_alpha = 0.25
        self.emb_dim = [8,8]
        self.max_episode = 1000  # 测试用，设置较小值
        self.init_std = np.sqrt(1e-1)
        self.gamma = 0.99
        self.algo_name = 'DQCAC'
        self.seed = 0

        # 学习率参数 (策略网络θ)
        self.theta_a = (10000**0.9) * 1e-3
        self.theta_b = 10000
        self.theta_c = 0.9
        
        # 学习率参数 (分位数估计Q)
        self.q_a = (10000**0.6) * 1e-2
        self.q_b = 10000
        self.q_c = 0.6
        
        # 学习率参数 (拉格朗日乘子λ)
        self.lambda_a = 0.3
        self.lambda_b = 5000
        self.lambda_c = 0.6

        # DQC-AC 特有参数
        self.outer_interval = 10          # λ更新间隔 (每几个episode更新一次)
        self.quantile_threshold = 4       # 分位数约束阈值 C (Q_α >= C)
        self.density_bandwidth = 0.01     # 密度估计带宽 δ (计算 Q(α±δ))
              
        # 设备
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


args = SimpleArgs()
random.seed(args.seed) 
np.random.seed(args.seed) 
torch.manual_seed(args.seed)

env = RiskSensitiveEnv(n=10)
agent_DQCAC = DQCAC(args, env)

# 启动训练
agent_DQCAC.train()

# 结束wandb监控
wandb.finish()

### 评估

In [ ]:
# QPO评估
from utils.evaluation import monte_carlo_evaluate
mean_QPO, quantile_QPO = monte_carlo_evaluate(agent_QPO, env, num_episodes=1000)

print(f"QPO mean:{mean_QPO:.4f}")
print(f"QPO quantile:{quantile_QPO:.4f}")

In [ ]:
# QCPO评估
from utils.evaluation import monte_carlo_evaluate
mean_QCPO, quantile_QCPO = monte_carlo_evaluate(agent_QCPO, env, num_episodes=1000)

print(f"QCPO mean:{mean_QCPO:.4f}")
print(f"QCPO quantile:{quantile_QCPO:.4f}")

In [ ]:
# DQC-AC 评估
from utils.evaluation import monte_carlo_evaluate
mean_DQCAC, quantile_DQCAC = monte_carlo_evaluate(agent_DQCAC, env, num_episodes=1000)

print(f"DQC-AC mean:{mean_DQCAC:.4f}")
print(f"DQC-AC quantile:{quantile_DQCAC:.4f}")

### 保存结果

In [ ]:
import json
import time
from pathlib import Path


def save_policy_result(agent, mean_value, quantile_value, algo_name, quantile_threshold=None):
    """保存策略评估结果并返回文件路径"""
    result_dir = Path("./notebook_results")
    result_dir.mkdir(parents=True, exist_ok=True)

    test_state = env.reset()
    if isinstance(test_state, tuple):
        test_state = test_state[0]
    test_action = agent.select_action(test_state.flatten())
    learned_risk = float(test_action[0])

    timestamp = time.strftime("%Y%m%d-%H%M%S")
    payload = {
        "algo_name": algo_name,
        "mean_return": float(mean_value),
        "quantile_return": float(quantile_value),
        "learned_risk": learned_risk,
        "q_alpha": args.q_alpha,
        "quantile_threshold": quantile_threshold,
        "density_estimate": args.density_estimate,
        "h_n": args.h_n if args.density_estimate else None,
        "nu": args.nu if not args.density_estimate else None,
        "outer_interval": args.outer_interval,
        "seed": args.seed,
        "timestamp": timestamp,
    }

    filename = result_dir / f"risk_{algo_name}_density{1 if args.density_estimate else 0}_th{args.quantile_threshold}_interval{args.outer_interval}_lambdaa{args.lambda_a}.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

    return filename


#qpo_file = save_policy_result(agent_QPO, mean_QPO, quantile_QPO, "QPO")
#print(f"QPO结果已保存至: {qpo_file}")



In [ ]:
qcpo_file = save_policy_result(
    agent_QCPO,
    mean_QCPO,
    quantile_QCPO,
    "QCPO",
    quantile_threshold=args.quantile_threshold,
)
print(f"QCPO结果已保存至: {qcpo_file}")



In [ ]:
# DQC-AC 结果保存
def save_dqcac_result(agent, mean_value, quantile_value):
    """保存DQC-AC评估结果"""
    result_dir = Path("./notebook_results")
    result_dir.mkdir(parents=True, exist_ok=True)

    test_state = env.reset()
    if isinstance(test_state, tuple):
        test_state = test_state[0]
    test_action = agent.select_action(test_state.flatten())
    learned_risk = float(test_action[0])

    timestamp = time.strftime("%Y%m%d-%H%M%S")
    payload = {
        "algo_name": "DQCAC",
        "mean_return": float(mean_value),
        "quantile_return": float(quantile_value),
        "learned_risk": learned_risk,
        "q_alpha": args.q_alpha,
        "quantile_threshold": args.quantile_threshold,
        "density_bandwidth": args.density_bandwidth,
        "outer_interval": args.outer_interval,
        "seed": args.seed,
        "timestamp": timestamp,
    }

    filename = result_dir / f"risk_DQCAC_th{args.quantile_threshold}_delta{args.density_bandwidth}_interval{args.outer_interval}.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

    return filename

dqcac_file = save_dqcac_result(agent_DQCAC, mean_DQCAC, quantile_DQCAC)
print(f"DQC-AC结果已保存至: {dqcac_file}")